---------

Comments
- 9% of location data are missing, but:
    - For active users we do not need location data
    - The missing data will average out (likely the 9% is MCAR)

--------
# Barter Deals dataset


- Construct the main predictor $apps\_after\_7\_days$
- Construct MAU/WAU (monthly and weekly active users, based on how many creators applied to a deal)
- Use location data to get estimates on the nr. of 'eligible' creators: the number of creators that are withina certain range of a (physical) deal

In [1]:
import numpy as np
import pandas as pd
from src import paths

In [ ]:
# Define your column lists
cols_deals = ['updated_at', 'deleted_at', 'id', 'legacy_partner_id', 'title', 'description', 'creators_requirement', 'hash_tags', 'status', 'deal_value', 'go_live_at', 'live_until', 'deal_type',
              'images', 'accepts_international', 'accepted_countries', 'social_requirement_type_id', 'product_name', 'schedule_type', 'schedule_model', 'legacy_id', 'tags', 'gender', 'featured_image', 'company_id', 'partner_id']
cols_comp = ['applicants_applications_count', 'cancelled_applications_count', 'company_locations', 'completed_applications_count', 'content_types', 'deal_created_at', 'deal_deleted_at', 'deal_id', 'deal_tags', 'deal_updated_at',
             'first_application_at', 'last_application_at', 'live_since', 'main_image', 'min_social_media_followers', 'pending_applications_count', 'planned_applications_count', 'rejected_applications_count', 'total_company_locations']

# 1. Load Deals
query_deals = f"SELECT {', '.join(cols_deals)} FROM public.deals;"
df_deals = pd.read_sql(query_deals, engine)

# 2. Load Deals Computed with built-in date parsing
# We add deal_id::text as deal_id_str directly in the SQL string
# query_comp = f"""
#     SELECT {', '.join(cols_comp)}, deal_id::text AS deal_id_str 
#     FROM public.deals_computed;
# """

query_comp = f"""
    SELECT {', '.join(cols_comp)}
    FROM public.deals_computed;
"""

df_deals_comp = pd.read_sql(
    query_comp,
    engine,
    parse_dates=['first_application_at', 'last_application_at']
)

# Merge
df_deals = pd.merge(df_deals,
                    df_deals_comp,
                    left_on='id', right_on='deal_id', how='left')

del df_deals_comp
# Text takes up a lot of memory, can drop since it is not needed
df_deals.drop(columns=['description', 'creators_requirement', 'schedule_model', 'main_image', 'images', 'deal_id'], inplace=True)

df_deals = df_deals.add_suffix('_deals')

# df_deals.rename(columns={'status': 'status_deals',
#                          'company_id': 'company_id_deals',
#                          'id': 'id_deals'}, inplace=True)

In [2]:
df_apps = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEAL_APPLICATIONS.parquet')
df_deals = pd.read_parquet(paths.RAW_DATA_DIR / 'BARTER_DEALS_CLEAN.parquet')

In [3]:
df_apps

,legacy_partner_id,legacy_location_id,deal_id,creator_id,rescheduling_deal_application_id,schedule_date,application_status,cancelled_at,cancelled_by,cancelled_by_user_id,...,created_at_creator,status_creator,socials,country,formatted_address,address_name,latitude_creator,longitude_creator,place_id,state_machine
0,279,335,019689e3-9b82-00c8-52df-2e22979a06ee,8162,<NA>,2024-04-12 00:00:00+00:00,rejected,NaT,<NA>,NaN,...,2024-03-14 12:44:44.122705+00:00,Active,"[{'id': 10749, 'social_media': 'Tiktok', 'foll...",Belgium,"Diest, Belgium",Diest,50.979461,5.055686,ChIJHbfyWcU4wUcR8E9NL6uZAAQ,NaN
1,279,335,019689e3-9b82-00c8-52df-2e22979a06ee,7787,<NA>,2024-03-26 00:00:00+00:00,cancelled,NaT,<NA>,NaN,...,2024-02-27 17:02:43.442790+00:00,Active,"[{'id': 10160, 'social_media': 'Instagram', 'f...",Netherlands,"Amsterdam, Netherlands",Amsterdam,52.367573,4.904139,ChIJVXealLU_xkcRja_At0z9AGY,NaN
2,279,335,019689e3-9b82-00c8-52df-2e22979a06ee,7827,<NA>,2024-05-29 00:00:00+00:00,accepted,NaT,<NA>,NaN,...,2024-02-28 12:30:36.449667+00:00,Active,"[{'id': 10227, 'social_media': 'Tiktok', 'foll...",Belgium,"Antwerp, Belgium",Antwerp,51.219930,4.414990,ChIJfYjDv472w0cRuIqogoRErz4,NaN
3,279,335,019689e3-9b82-00c8-52df-2e22979a06ee,358,<NA>,2024-07-19 00:00:00+00:00,accepted,NaT,<NA>,NaN,...,2023-10-05 21:15:17.888490+00:00,Active,"[{'id': 393, 'social_media': 'Instagram', 'fol...",Netherlands,"Amsterdam, Netherlands",Amsterdam,52.367573,4.904139,ChIJVXealLU_xkcRja_At0z9AGY,NaN
4,279,335,019689e3-9b82-00c8-52df-2e22979a06ee,343,<NA>,2024-05-20 00:00:00+00:00,accepted,NaT,<NA>,NaN,...,2023-10-04 18:00:42.491892+00:00,Active,"[{'id': 377, 'social_media': 'Instagram', 'fol...",<NA>,<NA>,<NA>,NaN,NaN,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269729,0,<NA>,019689e4-4bc6-00c8-6ff0-06aca342b058,42288,<NA>,2026-02-18 00:00:00+00:00,finished,NaT,<NA>,NaN,...,2026-01-27 08:18:43.376747+00:00,Active,"[{'id': 43692, 'social_media': 'Tiktok', 'foll...",Netherlands,"Diepenveen, Netherlands",Diepenveen,52.291527,6.148591,ChIJ8RGnLfDpx0cRZEZtD1kmqCk,NaN
269730,77,102,019689e4-4bc6-00c8-6ff0-06aca342b058,1577,<NA>,2024-06-29 00:00:00+00:00,finished,NaT,<NA>,NaN,...,2023-11-21 16:22:24.640519+00:00,Active,"[{'id': 1984, 'social_media': 'Tiktok', 'follo...",Netherlands,"Utrecht, Netherlands",Utrecht,52.091925,5.122957,ChIJNy3TOUNvxkcR6UqvGUz8yNY,NaN
269731,77,102,019689e4-4bc6-00c8-6ff0-06aca342b058,1577,<NA>,2024-07-06 00:00:00+00:00,finished,NaT,<NA>,NaN,...,2023-11-21 16:22:24.640519+00:00,Active,"[{'id': 1984, 'social_media': 'Tiktok', 'follo...",Netherlands,"Utrecht, Netherlands",Utrecht,52.091925,5.122957,ChIJNy3TOUNvxkcR6UqvGUz8yNY,NaN
269732,3454,<NA>,019b359e-7880-00c8-ba13-e73e44044ea9,36953,<NA>,2025-12-22 23:00:00+00:00,accepted,NaT,<NA>,NaN,...,2025-11-15 20:47:11.074719+00:00,Active,"[{'id': 41546, 'social_media': 'Tiktok', 'foll...",Netherlands,"Heemskerk, Netherlands",Heemskerk,52.514146,4.682137,ChIJE6QWKWBVz0cR52kAhnU_tpA,NaN


# Generate metrics

## Nr of active creators 


In [4]:
import pandas as pd
from collections import defaultdict

# 1. Sort the DataFrame by date (CRITICAL for a sliding window)
df_apps = df_apps.sort_values('created_at').reset_index(drop=True)

# 2. Extract to NumPy arrays for lightning-fast iteration
# Converting datetimes to nanoseconds makes comparisons incredibly fast
dates = df_apps['created_at'].astype('int64').values
influencers = df_apps['influencer_id'].values

# 3. Define your time windows in nanoseconds
ns_30_days = pd.Timedelta(days=30).value
ns_7_days = pd.Timedelta(days=7).value

def get_rolling_uniques(dates, influencers, window_ns):
    n = len(dates)
    results = [0] * n
    
    counts = defaultdict(int)
    unique_count = 0
    
    left = 0
    add_ptr = 0
    
    for i in range(n):
        current_time = dates[i]
        min_time = current_time - window_ns
        
        # EXPAND WINDOW: Add elements strictly LESS than the current row's time
        while add_ptr < n and dates[add_ptr] < current_time:
            inf = influencers[add_ptr]
            if counts[inf] == 0:
                unique_count += 1
            counts[inf] += 1
            add_ptr += 1
            
        # SHRINK WINDOW: Remove elements that fall outside the trailing window
        while left < add_ptr and dates[left] <= min_time:
            inf = influencers[left]
            counts[inf] -= 1
            if counts[inf] == 0:
                unique_count -= 1
            left += 1
            
        # Record the active unique count for this specific row
        results[i] = unique_count
        
    return results

# 4. Apply the optimized function
df_apps['active_last_month'] = get_rolling_uniques(dates, influencers, ns_30_days)
df_apps['active_last_week'] = get_rolling_uniques(dates, influencers, ns_7_days)

C:\Users\Wouter Barter\AppData\Local\Temp\ipykernel_17720\3882001998.py:28: RuntimeWarning: overflow encountered in scalar subtract
  min_time = current_time - window_ns
C:\Users\Wouter Barter\AppData\Local\Temp\ipykernel_17720\3882001998.py:28: RuntimeWarning: overflow encountered in scalar subtract
  min_time = current_time - window_ns


## Apps after 7 days

In [ ]:
def apps_after_n_days(row, n=7):
    subs = df_apps[df_apps['deal_id'] == row['deal_id']]
    if len(subs) != row['applicants_applications_count']:
        subs = subs[subs['deleted_at'].isna()]

    # "created_at" is the date on which the application log entry was created (application was made)
    days_since_live = (subs.created_at - row.live_since).dt.days
    return ((days_since_live >= 0) & (days_since_live <= n)).sum()

apps_after_7_days = df_deals.apply(apps_after_n_days, axis=1)
df_deals['apps_after_7_days'] = apps_after_7_days


------------

# Location computations tests

In [ ]:
import numpy as np

def calculate_haversine_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculates the great-circle distance between two points 
    on the Earth surface in kilometers using NumPy arrays.
    """
    # Earth radius in kilometers
    R = 6371.0 
    
    # Convert degrees to radians (NumPy trig functions require radians)
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    # Calculate differences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Apply the Haversine formula
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    
    # arcsin is mathematically equivalent to arctan2 for this and often slightly faster in numpy
    c = 2 * np.arcsin(np.sqrt(a)) 
    
    # Return distance in kilometers
    return R * c

1. Create spatial grid: distance between unique company locations and creator locations
2. Filter the creators that are within an eligible range (e.g., 25km)
    - Maybe explore whether I can come up with a metric that is a function of distance, e.g. weighted by distance from deal location
3. Per deal:
    - Total active (eligible) creators in last 7 days/30 days
        - Consider Pro creators? (creators with high follower count/ good reviews)


1. The Pre-Computed Spatial Grid (Calculate Once)

Do not calculate distances between deals and creators. Calculate distances between Unique Deal Locations and Unique Creator Locations.

Locations don't move. A coordinate in Amsterdam is always the same distance from a coordinate in Utrecht.

In [ ]:
# 1. Create a cross-join of ONLY the unique locations (1.3 million rows)
# df_unique_deals: 1710 rows (deal_loc_id, deal_lat, deal_lon)
# df_unique_creators: 773 rows (creator_loc_id, creator_lat, creator_lon)

df_spatial_grid = df_unique_deals.merge(df_unique_creators, how='cross')

# 2. Vectorize the Haversine distance formula here
# This gives you a permanent lookup table of distances between location IDs
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['deal_lat'], df_spatial_grid['deal_lon'],
    df_spatial_grid['creator_lat'], df_spatial_grid['creator_lon']
)

In [ ]:
# Calculate the distance for all 1.3 million combinations instantly
df_spatial_grid['distance_km'] = calculate_haversine_vectorized(
    df_spatial_grid['deal_lat'], 
    df_spatial_grid['deal_lon'],
    df_spatial_grid['creator_lat'], 
    df_spatial_grid['creator_lon']
)

# Optional: If you only care about creators within a specific radius (e.g., 50km), 
# drop the rest right now to save RAM before you join your time-series data!
df_spatial_grid = df_spatial_grid[df_spatial_grid['distance_km'] <= 50.0]

2. Dynamic Temporal Filtering (Calculate Often)

Now deal with the time aspect. You want to know who was active last week.§§

In [ ]:
# Filter your main creators dataframe (the one with 180k rows)
active_last_week = df_creators[df_creators['last_active'] >= '2026-02-12']

# Count how many ACTIVE creators are sitting at each unique location ID
# Result: A tiny dataframe of 773 rows showing available supply right now
supply_by_location = active_last_week.groupby('creator_loc_id').size().reset_index(name='active_creators')

3. The Final Join (Milliseconds)

Now, map that active supply onto your permanent spatial grid, and filter for deals that have creators within your desired radius (e.g., 25km).

In [ ]:
# Join the active counts to the spatial grid
eligible_supply = df_spatial_grid.merge(supply_by_location, on='creator_loc_id', how='left')

# Drop locations where no one is active
eligible_supply = eligible_supply.dropna(subset=['active_creators'])

# Filter for the radius you care about (e.g., within 25km)
deals_with_supply = eligible_supply[eligible_supply['distance_km'] <= 25]

# Group by deal to see total eligible creators nearby!
final_deal_supply = deals_with_supply.groupby('deal_loc_id')['active_creators'].sum()

In [6]:
df_locs = df_apps[~df_apps['company_location_id'].isna()]

In [11]:
creator_locs = df_locs[['longitude_creator', 'latitude_creator']]
partner_locs = df_locs[['longitude_partner', 'latitude_partner']]

In [13]:
partner_locs

,longitude_partner,latitude_partner
0,4.881908,52.358698
1,4.881908,52.358698
2,4.881908,52.358698
3,4.881908,52.358698
4,4.881908,52.358698
...,...,...
269727,4.887471,52.390218
269728,4.887471,52.390218
269729,4.887471,52.390218
269730,4.887471,52.390218


In [12]:
creator_locs

,longitude_creator,latitude_creator
0,5.055686,50.979461
1,4.904139,52.367573
2,4.414990,51.219930
3,4.904139,52.367573
4,NaN,NaN
...,...,...
269727,6.148591,52.291527
269728,6.148591,52.291527
269729,6.148591,52.291527
269730,5.122957,52.091925
